In [15]:
!pip install ultralytics rasterio shapely geopandas -q

In [16]:
import numpy as np
import rasterio
from ultralytics import YOLO
from shapely.geometry import box
import json, os

# ── CONFIG ──────────────────────────────────────────────────────────────
MODEL_PATH   = "/kaggle/input/models/pramcharanteja/best-model-samvita/other/default/2/best (9).pt"
VILLAGE_DIR  = "/kaggle/input/datasets/pramcharanteja/svamitva-14class-unified-train/punjab_bagga_vol1"
OUTPUT_PATH  = "/kaggle/working/bagga_detections.geojson"

CONF_THRESH = 0.30
NMS_IOU     = 0.3

CLASS_NAMES = {
    0:"Building_RCC", 1:"Building_Tiled", 2:"Building_Tin", 3:"Building_Other",
    4:"Road_Polygon", 5:"Waterbody_Polygon", 6:"Transformer", 7:"Overhead_Tank",
    8:"Utility_Polygon", 9:"Waterbody_Point"
}

# 🔥 slightly stricter thresholds
CLASS_CONF = {
    0: 0.40,
    1: 0.30,
    2: 0.40,
    3: 0.40,
    4: 0.30,
    5: 0.35,
    6: 0.25,
    7: 0.20,
    8: 0.25,
    9: 0.20,
}
# ────────────────────────────────────────────────────────────────────────

model = YOLO(MODEL_PATH)

tiles = [f for f in os.listdir(f"{VILLAGE_DIR}/images") if f.endswith(".tif")]
print(f"Found {len(tiles)} tiles")

features = []

for i, fname in enumerate(tiles):
    tif_path = f"{VILLAGE_DIR}/images/{fname}"

    with rasterio.open(tif_path) as src:
        transform = src.transform
        crs       = src.crs
        tile_arr  = src.read([1,2,3])
        tile_arr  = np.moveaxis(tile_arr, 0, -1)
        tile_arr  = tile_arr.astype(np.uint8)

    results = model(tile_arr, conf=CONF_THRESH, iou=NMS_IOU, verbose=False)[0]

    for det in results.boxes:
        x1, y1, x2, y2 = det.xyxy[0].cpu().numpy()
        conf = float(det.conf[0])
        cls  = int(det.cls[0])

        # ✅ per-class threshold
        if conf < CLASS_CONF[cls]:
            continue

        # pixel → geo
        geo_x1, geo_y1 = transform * (x1, y1)
        geo_x2, geo_y2 = transform * (x2, y2)

        geom = box(
            min(geo_x1, geo_x2), min(geo_y1, geo_y2),
            max(geo_x1, geo_x2), max(geo_y1, geo_y2)
        )

        features.append({
            "type": "Feature",
            "geometry": geom.__geo_interface__,
            "properties": {
                "class_id": cls,
                "class_name": CLASS_NAMES[cls],
                "confidence": round(conf, 4),
                "source_tile": fname,
            }
        })

    if i % 200 == 0:
        print(f"{i}/{len(tiles)} tiles done, {len(features)} detections so far")

print(f"\nTotal BEFORE NMS: {len(features)}")

Found 1435 tiles
0/1435 tiles done, 1 detections so far
200/1435 tiles done, 197 detections so far
400/1435 tiles done, 382 detections so far
600/1435 tiles done, 568 detections so far
800/1435 tiles done, 744 detections so far
1000/1435 tiles done, 958 detections so far
1200/1435 tiles done, 1141 detections so far
1400/1435 tiles done, 1324 detections so far

Total BEFORE NMS: 1349


In [17]:
from torchvision.ops import nms
import torch

print("\nRunning cross-tile NMS...")

all_boxes = []
all_scores = []
all_classes = []

for feat in features:
    coords = feat["geometry"]["coordinates"][0]
    xs = [c[0] for c in coords]
    ys = [c[1] for c in coords]

    all_boxes.append([min(xs), min(ys), max(xs), max(ys)])
    all_scores.append(feat["properties"]["confidence"])
    all_classes.append(feat["properties"]["class_id"])

boxes = torch.tensor(all_boxes, dtype=torch.float32)
scores = torch.tensor(all_scores, dtype=torch.float32)

keep_indices = set()

for cls in set(all_classes):
    idx = [i for i, c in enumerate(all_classes) if c == cls]
    if not idx:
        continue

    idx_t = torch.tensor(idx)
    keep = nms(boxes[idx_t], scores[idx_t], iou_threshold=0.3)

    for k in keep.tolist():
        keep_indices.add(idx[k])

features = [features[i] for i in sorted(keep_indices)]

print(f"After NMS: {len(features)}")


Running cross-tile NMS...
After NMS: 1347


In [18]:
import numpy as np

print("\nApplying spatial filtering...")

centroids = []

for feat in features:
    coords = feat["geometry"]["coordinates"][0]
    cx = np.mean([c[0] for c in coords])
    cy = np.mean([c[1] for c in coords])
    centroids.append((cx, cy))

# compute center
center_x = np.mean([c[0] for c in centroids])
center_y = np.mean([c[1] for c in centroids])

# compute distances
distances = [
    ((cx - center_x)**2 + (cy - center_y)**2)**0.5
    for (cx, cy) in centroids
]

# 🔥 adaptive threshold instead of fixed value
threshold = np.percentile(distances, 85)  # keep central 85%

filtered = []

for feat, dist in zip(features, distances):
    if dist <= threshold:
        filtered.append(feat)

features = filtered

print(f"After spatial filtering: {len(features)} (threshold={threshold})")


Applying spatial filtering...
After spatial filtering: 1145 (threshold=363.4529832019956)


In [19]:
import pyproj
from shapely.ops import transform as shp_transform
from shapely.geometry import shape
from collections import Counter
import json

project = pyproj.Transformer.from_crs(
    str(crs), "EPSG:4326", always_xy=True
).transform

features_wgs84 = []

for feat in features:
    geom = shape(feat["geometry"])
    geom_wgs84 = shp_transform(project, geom)

    features_wgs84.append({
        "type": "Feature",
        "geometry": geom_wgs84.__geo_interface__,
        "properties": feat["properties"]
    })

geojson = {
    "type": "FeatureCollection",
    "features": features_wgs84
}

with open(OUTPUT_PATH, "w") as f:
    json.dump(geojson, f, indent=2)

# summary
counts = Counter(f["properties"]["class_name"] for f in features_wgs84)

print(f"\nSaved {len(features_wgs84)} detections → {OUTPUT_PATH}\n")

for cls, count in sorted(counts.items()):
    print(f"{cls}: {count}")

# sanity check
if features_wgs84:
    g = features_wgs84[0]["geometry"]["coordinates"][0]
    print(f"\nFirst bbox coords: {g[0]}")


Saved 1145 detections → /kaggle/working/bagga_detections.geojson

Building_Other: 9
Building_RCC: 640
Building_Tiled: 1
Building_Tin: 350
Road_Polygon: 76
Transformer: 2
Waterbody_Polygon: 67

First bbox coords: (75.15770095945805, 31.722203465998984)


In [20]:
# import numpy as np
# import rasterio
# from ultralytics import YOLO
# from shapely.geometry import box
# import json, os
# from PIL import Image

# # ── CONFIG ──────────────────────────────────────────────────────────────
# MODEL_PATH   = "/kaggle/input/models/pramcharanteja/best-model-samvita/other/default/2/best (9).pt"
# VILLAGE_DIR  = "/kaggle/input/datasets/pramcharanteja/svamitva-14class-unified-train/punjab_bagga_vol1"
# OUTPUT_PATH  = "/kaggle/working/bagga_detections.geojson"
# CONF_THRESH = 0.30   # was 0.25, higher = fewer but more confident detections
# NMS_IOU     = 0.3    # was 0.5, lower = more aggressive suppression of overlaps

# CLASS_NAMES = {
#     0:"Building_RCC", 1:"Building_Tiled", 2:"Building_Tin", 3:"Building_Other",
#     4:"Road_Polygon", 5:"Waterbody_Polygon", 6:"Transformer", 7:"Overhead_Tank",
#     8:"Utility_Polygon", 9:"Waterbody_Point"
# }
# CLASS_CONF = {
#     0: 0.35,
#     1: 0.30,
#     2: 0.35,
#     3: 0.35,
#     4: 0.30,
#     5: 0.25,
#     6: 0.25,
#     7: 0.20,
#     8: 0.25,
#     9: 0.20,
# }
# # ────────────────────────────────────────────────────────────────────────

# model = YOLO(MODEL_PATH)
# tiles = [f for f in os.listdir(f"{VILLAGE_DIR}/images") if f.endswith(".tif")]
# print(f"Found {len(tiles)} tiles")

# features = []

# for i, fname in enumerate(tiles):
#     tif_path = f"{VILLAGE_DIR}/images/{fname}"
#     with rasterio.open(tif_path) as src:
#         transform = src.transform
#         crs       = src.crs
#         tile_arr  = src.read([1,2,3])          # (3, H, W)
#         tile_arr  = np.moveaxis(tile_arr, 0, -1)  # (H, W, 3)
#         tile_arr  = tile_arr.astype(np.uint8)

#     results = model(tile_arr, conf=CONF_THRESH, iou=0.3, verbose=False)[0]
    
#     for det in results.boxes:
#         x1, y1, x2, y2 = det.xyxy[0].cpu().numpy()
#         conf = float(det.conf[0])
#         cls  = int(det.cls[0])

#         # pixel → geographic using this tile's affine transform
#         geo_x1, geo_y1 = transform * (x1, y1)
#         geo_x2, geo_y2 = transform * (x2, y2)
#         geom = box(
#             min(geo_x1, geo_x2), min(geo_y1, geo_y2),
#             max(geo_x1, geo_x2), max(geo_y1, geo_y2)
#         )
#         features.append({
#             "type": "Feature",
#             "geometry": geom.__geo_interface__,
#             "properties": {
#                 "class_id":   int(cls),
#                 "class_name": CLASS_NAMES[int(cls)],
#                 "confidence": round(float(conf), 4),
#                 "source_tile": fname,
#             }
#         })

#     if i % 200 == 0:
#         print(f"  {i}/{len(tiles)} tiles done, {len(features)} detections so far")

# print(f"\nTotal detections: {len(features)}")

In [21]:
# import pyproj
# from shapely.ops import transform as shp_transform
# from shapely.geometry import shape
# import functools

# # reproject from source CRS → WGS84
# project = pyproj.Transformer.from_crs(
#     str(crs), "EPSG:4326", always_xy=True
# ).transform

# features_wgs84 = []
# for feat in features:
#     geom = shape(feat["geometry"])
#     geom_wgs84 = shp_transform(project, geom)
#     features_wgs84.append({
#         "type": "Feature",
#         "geometry": geom_wgs84.__geo_interface__,
#         "properties": feat["properties"]
#     })

# geojson = {
#     "type": "FeatureCollection",
#     "features": features_wgs84
# }
# with open(OUTPUT_PATH, "w") as f:
#     json.dump(geojson, f, indent=2)

# # summary
# from collections import Counter
# counts = Counter(f["properties"]["class_name"] for f in features_wgs84)
# print(f"Saved {len(features_wgs84)} detections → {OUTPUT_PATH}\n")
# for cls, count in sorted(counts.items()):
#     print(f"  {cls}: {count}")

# # sanity check — print first bbox
# if features_wgs84:
#     g = features_wgs84[0]["geometry"]["coordinates"][0]
#     print(f"\nFirst bbox coords (should be lat/lon): {g[0]}")